# **Streaming Agents**

## **What's Covered?**
1. Streaming Agent's Response
    - Streaming Agent Progress
    - Streaming LLM Response
    - Streaming Multiple Modes

## **Introduction to Streaming**

### **Why Streaming?**
Interactive applications like chatbots or customer service agents can suffer from high latency. To get data to the user as soon as it is available you can deploy streaming.

By displaying output progressively, even before a complete response is ready, streaming significantly improves user experience (UX), particularly when dealing with the latency of LLMs.

### **Streaming Agent's Response**

LangChain’s streaming system lets you surface live feedback from agent runs to your application. Following is possible with LangChain Streaming:
1. Stream agent progress - i.e. get state updates after each agent step
2. Stream LLM tokens - i.e. stream language model tokens as they’re generated
3. Stream reasoning/thinking tokens - i.e. surface model reasoning as it’s generated
4. Stream custom updates - i.e. emit user-defined signals
5. Stream multiple modes - i.e. choose from `updates` (agent progress), `messages` (LLM tokens + metadata), or `custom` (arbitrary user data).

### **Supported Streaming Modes**
There are several streaming modes frequently used with the LangChain agents:
1. `messages`: Streams tuples of (token, metadata) from any graph nodes where an LLM is invoked
2. `updates`: Streams state updates after each agent step. If multiple updates are made in the same step (e.g., multiple nodes are run), those updates are streamed separately
3. `custom`: Streams custom data from inside your graph nodes using the stream writer

### **Syntax**
```python
for chunk in agent.stream(  
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
):
    # do something
    pass
```

## **Without Streaming**

In [1]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

openai_chat_model = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model="gpt-4o-mini",
    temperature=0.0
)

In [3]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model=openai_chat_model,
    tools=[get_weather],
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather in SF?"}]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is the weather in SF?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_WQ2yF9ltFQLSM0SkrbU8Xz15)
 Call ID: call_WQ2yF9ltFQLSM0SkrbU8Xz15
  Args:
    city: San Francisco
================================= Tool Message =================================
Name: get_weather

It's always sunny in San Francisco!
================================== Ai Message ==================================

The weather in San Francisco is sunny!


## **Stream LLM Tokens**
Messages stream data token by token. This mode produce the lowest latency possible for the end user. This is perfect for interactive applications like chatbots or customer support agents. 

To stream tokens as they are produced by the LLM, use stream_mode="messages". Below you can see the output of the agent streaming tool calls and the final response.

Notice that, in the following output, it prints the tool message first and later the AI Message token by token.

In [18]:
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="messages",
):
    print(f"step: {metadata['langgraph_node']} | step #: {metadata['langgraph_step']}")
    if token.content:
        print(token.content, end="\n")

step: model | step #: 1
step: model | step #: 1
step: model | step #: 1
step: model | step #: 1
step: model | step #: 1
step: model | step #: 1
step: model | step #: 1
step: model | step #: 1
step: model | step #: 1
step: model | step #: 1
step: tools | step #: 2
It's always sunny in San Francisco!
step: model | step #: 3
step: model | step #: 3
The
step: model | step #: 3
 weather
step: model | step #: 3
 in
step: model | step #: 3
 San
step: model | step #: 3
 Francisco
step: model | step #: 3
 is
step: model | step #: 3
 always
step: model | step #: 3
 sunny
step: model | step #: 3
!
step: model | step #: 3
step: model | step #: 3
step: model | step #: 3


## **Streaming Agent Progress**

Let's look at two ways to stream agent output after each step.

1. stream_mode="updates"
2. stream_mode="values"

To stream agent progress, use the stream or astream methods with **stream_mode="updates"**. This emits an event after every agent step.

**Note:** Emit only the node or task names and updates returned by the nodes or tasks after each step. If multiple updates are made in the same step (e.g. multiple nodes are run) then those updates are emitted separately.

In [10]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'call_29ahi9S7QDb2IYdAzKmFc86Q'}]
step: tools
content: [{'type': 'text', 'text': "It's always sunny in San Francisco!"}]
step: model
content: [{'type': 'text', 'text': 'The weather in San Francisco is always sunny!'}]


### **stream_mode="values"**

Value streaming mode returns data after each step in the agent loop. So we see updates after a model call, or a tool call, etc...

In [19]:
for chunk in agent.stream(  
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="values",
):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

What is the weather in SF?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_qYAXB5yRrN68S55eps6ruih2)
 Call ID: call_qYAXB5yRrN68S55eps6ruih2
  Args:
    city: San Francisco
================================= Tool Message =================================
Name: get_weather

It's always sunny in San Francisco!
================================== Ai Message ==================================

The weather in San Francisco is always sunny!


## **Stream Custom Updates**

Tools can stream too. To stream updates from tools as they are executed, you can use **get_stream_writer**.

In [20]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()  
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"

agent = create_agent(
    model=openai_chat_model,
    tools=[get_weather],
)

In [21]:
for chunk in agent.stream(  
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="custom",
):
    print(chunk, end="\n")

Looking up data for city: San Francisco
Acquired data for city: San Francisco
